# Steam description cleanup and translation queue preparation

This notebook starts from `games_semi_processed.csv`, applies deterministic code-based cleanup, detects confidently non-English description segments, and exports the affected games into **nine balanced CSV files** under `Translation Needed`. It does not translate descriptions or create an English-only dataset.

Language detection is run on likely multilingual descriptions. A game is queued when at least one segment of 25 or more letters is confidently identified as non-English. Short or low-confidence segments are recorded as uncertain and do not alone put a game in the queue. Each batch includes AppID, Name, the cleaned description, and language-detection details so translated results can later be matched back by AppID.

The final audit checks the nine exported files for row counts, duplicate/missing AppIDs, emojis, HTML, mojibake markers, replacement characters, and non-ASCII letters. Run cells in order; this notebook is not run automatically.

In [ ]:
from pathlib import Path
from html import unescape
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import regex as unicode_regex
from bs4 import BeautifulSoup
from ftfy import fix_text
from langdetect import detect_langs, DetectorFactory
from IPython.display import display

DetectorFactory.seed = 0

# Find the Data Prep folder whether Jupyter starts at the project root or inside Data Prep.
input_candidates = [
    Path('games_semi_processed.csv'),
    Path('Data Prep/games_semi_processed.csv'),
    Path('../Data Prep/games_semi_processed.csv'),
]
input_path = next((path for path in input_candidates if path.is_file()), None)
if input_path is None:
    raise FileNotFoundError('Could not find games_semi_processed.csv. Checked: ' + ', '.join(map(str, input_candidates)))
translation_needed_dir = input_path.parent / 'Translation Needed'
translation_done_dir = input_path.parent / 'Translation Done'
NUM_BATCHES = 9
DESCRIPTION_COLUMN = 'About the game'
APP_ID_COLUMN = 'AppID'
NAME_COLUMN = 'Name'
MIN_SEGMENT_LETTERS = 25
MIN_LANGUAGE_CONFIDENCE = 0.80

print('Input:', input_path.resolve())
print('Translation-needed folder:', translation_needed_dir.resolve())
print('Translation-done folder:', translation_done_dir.resolve())
print('Number of batches:', NUM_BATCHES)

In [ ]:
# Deterministic code-based cleanup. Preserve the source description for audit.
BLOCK_HTML_RE = unicode_regex.compile(
    r'</?\s*(?:p|div|ul|ol|li|br|h[1-6]|hr|table|tr|td|th|section|article)\b[^>]*>',
    flags=unicode_regex.IGNORECASE,
)
EMOJI_RE = unicode_regex.compile(
    r'(?:\p{Extended_Pictographic}(?:\p{Emoji_Modifier})?(?:\uFE0F|\uFE0E)?'
    r'(?:\u200D\p{Extended_Pictographic}(?:\p{Emoji_Modifier})?(?:\uFE0F|\uFE0E)?)*)'
    r'|(?:\p{Regional_Indicator}{2})|(?:[#*0-9]\uFE0F?\u20E3)'
)

MOJIBAKE_MARKERS = ('Ã', 'Â', 'â€', 'ðŸ', 'ï¿½')
HTML_ENTITY_RE = re.compile(r'&(?:[A-Za-z][A-Za-z0-9]+|#\d+|#x[0-9A-Fa-f]+);')
CJK_RE = unicode_regex.compile(r'(?:\p{Script=Han}|\p{Script=Hiragana}|\p{Script=Katakana}|\p{Script=Hangul})')
SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+|(?<=[。！？])|[\r\n]+')
NON_ENGLISH_MARKERS = set(
    'le les des est avec pour dans une jeu du et la los las del para pero juego una con que '
    'nicht der die das und ist ein spiel mit il lo gli che gioco per não você het een van zijn voor niet'.split()
)


def clean_description(value):
    text = '' if pd.isna(value) else str(value)
    original = text
    entity_decode_passes = 0
    for _ in range(2):
        decoded = unescape(text)
        if decoded == text:
            break
        text = decoded
        entity_decode_passes += 1

    mojibake_repaired = any(marker in text for marker in MOJIBAKE_MARKERS)
    if mojibake_repaired:
        text = fix_text(text)

    html_stripped = bool(BLOCK_HTML_RE.search(text))
    if html_stripped:
        soup = BeautifulSoup(text, 'html.parser')
        for element in soup(['script', 'style']):
            element.decompose()
        text = soup.get_text(separator=' ', strip=True)

    emoji_count = len(EMOJI_RE.findall(text))
    if emoji_count:
        text = EMOJI_RE.sub(' ', text)

    replacement_count = text.count('\ufffd')
    if replacement_count:
        text = text.replace('\ufffd', ' ')

    control_count = 0
    output_chars = []
    for char in text:
        if unicodedata.category(char) == 'Cc' and char not in '\t\r\n':
            control_count += 1
            output_chars.append(' ')
        else:
            output_chars.append(char)
    text = ''.join(output_chars).replace('\r', ' ').replace('\n', ' ').replace('\t', ' ')
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return {
        'cleaned': text,
        'entity_decode_passes': entity_decode_passes,
        'html_formatting_stripped': html_stripped,
        'mojibake_repaired': mojibake_repaired,
        'emojis_removed': emoji_count,
        'replacement_characters_removed': replacement_count,
        'control_characters_removed': control_count,
        'text_changed': text != original,
    }


def likely_needs_language_detection(text):
    text = '' if pd.isna(text) else str(text)
    if CJK_RE.search(text):
        return True
    letters = [char for char in text if char.isalpha()]
    non_ascii_letters = sum(ord(char) > 127 for char in letters)
    if non_ascii_letters >= 4 and non_ascii_letters / max(1, len(letters)) >= 0.02:
        return True
    words = set(unicode_regex.findall(r'\p{L}+', text.lower()))
    return len(words & NON_ENGLISH_MARKERS) >= 2


def count_letters(text):
    return sum(char.isalpha() for char in text)


def analyze_language_segments(text):
    text = '' if pd.isna(text) else str(text)
    segments = [part.strip() for part in SENTENCE_SPLIT_RE.split(text) if part.strip()]
    total_letters = sum(count_letters(part) for part in segments)
    non_english_letters = 0
    non_english_segments = []
    uncertain_segments = []
    for segment in segments:
        letters = count_letters(segment)
        if letters < MIN_SEGMENT_LETTERS:
            uncertain_segments.append(segment)
            continue
        try:
            predictions = detect_langs(segment[:1500])
        except Exception:
            uncertain_segments.append(segment)
            continue
        if not predictions or predictions[0].prob < MIN_LANGUAGE_CONFIDENCE:
            uncertain_segments.append(segment)
            continue
        if predictions[0].lang != 'en':
            non_english_letters += letters
            non_english_segments.append(segment)
    share = non_english_letters / total_letters if total_letters else 0.0
    uncertain_share = sum(count_letters(part) for part in uncertain_segments) / total_letters if total_letters else 0.0
    return {
        'segments': segments,
        'non_english_segments': non_english_segments,
        'uncertain_segments': uncertain_segments,
        'non_english_share': share,
        'uncertain_share': uncertain_share,
    }

# Read the semi-processed source afresh. This avoids inheriting prior filtering decisions.
df = pd.read_csv(input_path, encoding='utf-8-sig', low_memory=False, dtype={APP_ID_COLUMN: 'string'})
required = {APP_ID_COLUMN, NAME_COLUMN, DESCRIPTION_COLUMN}
missing = sorted(required - set(df.columns))
if missing:
    raise KeyError(f'Required input columns are missing: {missing}')
if 'Description original' not in df.columns:
    df['Description original'] = df[DESCRIPTION_COLUMN]

clean_records = [clean_description(value) for value in df[DESCRIPTION_COLUMN]]
clean_metadata = pd.DataFrame(clean_records, index=df.index)
df[DESCRIPTION_COLUMN] = clean_metadata['cleaned']
for column in [
    'entity_decode_passes', 'html_formatting_stripped', 'mojibake_repaired',
    'emojis_removed', 'replacement_characters_removed', 'control_characters_removed', 'text_changed',
]:
    df[f'Description {column.replace("_", " ")}'] = clean_metadata[column]

print(f'Games loaded: {len(df):,}')
print(f'Descriptions changed by code cleanup: {int(clean_metadata["text_changed"].sum()):,}')
print(f'Likely multilingual candidates: {int(df[DESCRIPTION_COLUMN].map(likely_needs_language_detection).sum()):,}')


In [ ]:
# Detect confident non-English description content, then split it from descriptions that need no translation.
share_values = []
non_english_segments_by_row = []
uncertain_segments_by_row = []
uncertain_share_values = []
candidate_mask = df[DESCRIPTION_COLUMN].map(likely_needs_language_detection)
for position, (idx, text) in enumerate(df[DESCRIPTION_COLUMN].items(), start=1):
    if candidate_mask.at[idx]:
        result = analyze_language_segments(text)
        share_values.append((idx, result['non_english_share']))
        non_english_segments_by_row.append((idx, result['non_english_segments']))
        uncertain_segments_by_row.append((idx, result['uncertain_segments']))
        uncertain_share_values.append((idx, result['uncertain_share']))
    else:
        share_values.append((idx, 0.0))
        non_english_segments_by_row.append((idx, []))
        uncertain_segments_by_row.append((idx, []))
        uncertain_share_values.append((idx, 0.0))
    if position % 5000 == 0:
        print(f'Language analysis progress: {position:,}/{len(df):,}')

share_by_index = dict(share_values)
non_english_by_index = dict(non_english_segments_by_row)
uncertain_by_index = dict(uncertain_segments_by_row)
uncertain_share_by_index = dict(uncertain_share_values)
df['Description non-English letter share'] = pd.Series(share_by_index)
df['Description uncertain letter share'] = pd.Series(uncertain_share_by_index)
df['Description non-English sentences'] = pd.Series({i: json.dumps(v, ensure_ascii=False) for i, v in non_english_by_index.items()})
df['Description uncertain sentences'] = pd.Series({i: json.dumps(v, ensure_ascii=False) for i, v in uncertain_by_index.items()})

# Keep source order so the final audit notebook can restore the original row order.
df['__source_row'] = df.index
empty_description_mask = df[DESCRIPTION_COLUMN].fillna('').astype(str).str.strip().eq('')
empty_descriptions_dropped = int(empty_description_mask.sum())
df = df.loc[~empty_description_mask].copy()
translation_mask = df['Description non-English letter share'].gt(0)
translation_rows = df.loc[translation_mask].copy()
no_translation_rows = df.loc[~translation_mask].copy()

for label, subset in [('translation-needed', translation_rows), ('no-translation-needed', no_translation_rows)]:
    if subset[APP_ID_COLUMN].isna().any():
        raise ValueError(f'{label} rows contain missing AppIDs.')
    if subset[APP_ID_COLUMN].duplicated().any():
        raise ValueError(f'{label} rows contain duplicate AppIDs; resolve these before export.')

translation_needed_dir.mkdir(parents=True, exist_ok=True)
translation_done_dir.mkdir(parents=True, exist_ok=True)
no_translation_path = translation_done_dir / 'no_translation_needed.csv'
no_translation_rows.to_csv(no_translation_path, index=False, encoding='utf-8-sig')
print(f'Saved {no_translation_path.name}: {len(no_translation_rows):,} games')

batch_files = []
for batch_number, batch_indices in enumerate(np.array_split(translation_rows.index.to_numpy(), NUM_BATCHES), start=1):
    batch = translation_rows.loc[batch_indices].copy()
    batch_path = translation_needed_dir / f'translation_batch_{batch_number:02d}_of_{NUM_BATCHES:02d}.csv'
    batch.to_csv(batch_path, index=False, encoding='utf-8-sig')
    batch_files.append(batch_path)
    print(f'Saved {batch_path.name}: {len(batch):,} games')

print(f'Games loaded: {len(clean_metadata):,}')
print(f'Games dropped because descriptions were empty: {empty_descriptions_dropped:,}')
print(f'Likely multilingual candidates analyzed: {int(candidate_mask.sum()):,}')
print(f'Games queued for translation: {len(translation_rows):,}')
print(f'Games needing no translation: {len(no_translation_rows):,}')
print(f'Uncertain descriptions queued for translation: {int(translation_rows["Description uncertain letter share"].gt(0).sum()):,}')
print(f'Average games per translation batch: {len(translation_rows) / NUM_BATCHES:.1f}')
